<a href="https://colab.research.google.com/github/gow504/-RAG-Based-PDF-Question-Answering-System/blob/main/question_answer_using_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install langchain


In [3]:
!pip install langchain-chroma
!pip install langchain-huggingface
!pip install langchain-groq
!pip install pypdf
!pip install torch torchvision torchaudio

In [4]:
!pip install transformers==4.57.1
!pip install sentence-transformers==5.1.2
!pip install accelerate==1.10.1

In [5]:
from langchain_groq import ChatGroq
from langchain_chroma import Chroma
from langchain_core.prompts import HumanMessagePromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_core.messages import SystemMessage
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
import os

/tmp/ipykernel_16026/3858643317.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
from google.colab import userdata
os.environ['GROQ_API_KEY'] = userdata.get("GROQ_API_KEY")
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature = 0
)

In [7]:
from google.colab import files
uploaded = files.upload()
print("uploaded successfully")

# Extract the correct string path of the uploaded file
filename = list(uploaded.keys())[0]
pdf_file_path = f"/content/{filename}"
print(f"File path to use: {pdf_file_path}")

Saving JNTUK R20 ML UNIT-V.pdf to JNTUK R20 ML UNIT-V.pdf
uploaded successfully
File path to use: /content/JNTUK R20 ML UNIT-V.pdf


In [8]:
# Load and split the PDF document into chunks
document_loader = PyPDFLoader(pdf_file_path)
# documents = document_loader.load_and_split()
documents = document_loader.load()

In [9]:
documents

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-04-21T19:18:42+05:30', 'author': 'dodda venkatareddy', 'moddate': '2023-04-21T19:18:42+05:30', 'source': '/content/JNTUK R20 ML UNIT-V.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}, page_content='Unit V \nNeural Networks and Deep Learning \nIntroduction to Artificial Neural Networks with Keras, Implementing MLPs with Keras, Installing TensorFlow 2, \nLoading and Preprocessing Data with TensorFlow \n………………………………………………………………………………………………………………………………... \nQ) Introduction to Artificial Neural Networks with Keras \n1. Neurons and biological motivation \nWhat is an Artificial Neural Network? \n• A Neural Network is a system designed to operate like a human brain. Human information processing \ntakes place through the interaction of many billions of neurons connected to each other sending signals \nto other neurons. \n• Similarly, a Neural Network is a network of artificial

In [ ]:
len(documents)

In [10]:
text_splitter = CharacterTextSplitter(
    chunk_size=2000,
    chunk_overlap=500
)

text_chunks = text_splitter.split_documents(documents)

In [11]:
text_chunks

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2023-04-21T19:18:42+05:30', 'author': 'dodda venkatareddy', 'moddate': '2023-04-21T19:18:42+05:30', 'source': '/content/JNTUK R20 ML UNIT-V.pdf', 'total_pages': 19, 'page': 0, 'page_label': '1'}, page_content='Unit V \nNeural Networks and Deep Learning \nIntroduction to Artificial Neural Networks with Keras, Implementing MLPs with Keras, Installing TensorFlow 2, \nLoading and Preprocessing Data with TensorFlow \n………………………………………………………………………………………………………………………………... \nQ) Introduction to Artificial Neural Networks with Keras \n1. Neurons and biological motivation \nWhat is an Artificial Neural Network? \n• A Neural Network is a system designed to operate like a human brain. Human information processing \ntakes place through the interaction of many billions of neurons connected to each other sending signals \nto other neurons. \n• Similarly, a Neural Network is a network of artificial

In [12]:
embeddings = HuggingFaceEmbeddings(model_name="sangmini/msmarco-cotmae-MiniLM-L12_en-ko-ja")

Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
/usr/local/lib/python3.13/dist-packages/jax/_src/cloud_tpu_init.py:90: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authen

In [13]:
# Initialize Chroma vector database
persist_directory = r"D:\vectorstore1"  # Replace with your actual persistence directory path
db = Chroma.from_documents(
    documents=text_chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)

In [14]:
retriever = db.as_retriever()

In [15]:
# Initialize the RetrievalQA chain with your LLM and the Chroma retriever
db_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",
    retriever=retriever
)

In [16]:
def retrieve_from_db(query: str) -> str:
    try:
        db_context = db_chain(query)
        db_context = db_context['result'].strip()
        return db_context
    except Exception as e:
        print(f"Error: {e}")
        return "An error occurred while retrieving data from the database."


In [18]:
def generate(query: str) -> str:
    db_context = retrieve_from_db(query)

    system_message = """
            You are a RAG assistant for 'The Data Science Handbook'.

            Rules:
            1. Answer ONLY using the retrieved context.
            2. Never use your own knowledge.
            3. If the retrieved context does not directly answer the user's question, respond exactly:
               "This question is outside the scope of the provided document."
            4. Do not infer, assume, or guess.
            5. If the answer is only partially supported by the context, respond:
               "I don't have enough information in the provided document."
            6. Be concise and friendly.
        """

    human_qry_template = HumanMessagePromptTemplate.from_template(
        """Input:
        {human_input}

        Context:
        {db_context}

        Output:
        """
    )
    messages = [
        SystemMessage(content=system_message),
        human_qry_template.format(human_input=query, db_context=db_context)
    ]
    try:
        # Get the response from the LLM using the invoke method
        response = llm.invoke(messages).content
        return response
    except Exception as e:
        print(f"Error: {e}")
        return "An error occurred while generating the response."

# Example of usage
query_input = input("Enter your query: ")
response_data = generate(query_input)
print(f"Response: {response_data}")

Enter your query: activation function
Response: **Activation function**  
In an artificial neuron it is the “cell body” that decides whether the neuron fires.  
It takes the weighted sum of inputs (plus bias) and maps it to an output value.  
Typical Keras syntax: `activation='relu'`, `activation='sigmoid'`, etc.  

Examples from the table:  

| Function | Formula | Biological analogue | Why it matters |
|----------|---------|---------------------|----------------|
| **Step** | `φ(z) = 1 if z ≥ θ else 0` | Hard firing threshold | Non‑differentiable → not used for training |
| **Sigmoid** | `φ(z) = 1/(1+e⁻z)` | Smooth firing probability | Differentiable, bounded → early neural nets |
| **Tanh** | `φ(z) = tanh(z)` | Symmetric firing | Zero‑centered → better gradients |
| **ReLU** | `φ(z) = max(0, z)` | “All‑or‑nothing” with linear growth | Fast, sparse, mitigates vanishing gradients |
| **LeakyReLU / ELU / SELU** | Variants that allow a small negative slope | Prevents “dying” neurons | I